In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

df = pd.read_csv('penguins_lter.csv')

features = ['Culmen Length (mm)', 'Culmen Depth (mm)', 'Flipper Length (mm)', 'Body Mass (g)']
df_clean = df.dropna(subset=features + ['Species']).copy()

df_clean['Species_Short'] = df_clean['Species'].apply(lambda x: x.split()[0])

X = df_clean[features]
y = df_clean['Species_Short']

print(f"จำนวนข้อมูลทั้งหมดหลังทำความสะอาด: {df_clean.shape[0]} ตัวอย่าง")
print(f"การกระจายตัวของคลาส:\n{y.value_counts()}\n")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

kernels = ['linear', 'poly', 'rbf']
results = {}

print("="*60)
print("สรุปผลการทดสอบ SVM แต่ละ Kernel Function")
print("="*60)

for k in kernels:
    model = SVC(kernel=k, C=1.0, gamma='scale', random_state=42)
    model.fit(X_train_scaled, y_train)
    
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    
    results[k] = {
        'model': model,
        'accuracy': acc,
        'predictions': y_pred
    }
    
    print(f"Kernel: {k.upper():<10} | Accuracy: {acc * 100:.2f}% ({acc:.4f})")
    print(f"จำนวน Support Vectors: {model.n_support_} (ต่อแต่ละคลาส)")
    print("-" * 60)

print("\nClassification Report สำหรับ RBF Kernel:")
print(classification_report(y_test, results['rbf']['predictions']))

comparison_df = pd.DataFrame({
    'Actual': y_test.values[:10],
    'Pred_Linear': results['linear']['predictions'][:10],
    'Pred_Poly': results['poly']['predictions'][:10],
    'Pred_RBF': results['rbf']['predictions'][:10]
})
print("ตัวอย่างผลการทำนาย (10 ข้อมูลแรกใน Test Set):")
print(comparison_df)

จำนวนข้อมูลทั้งหมดหลังทำความสะอาด: 342 ตัวอย่าง
การกระจายตัวของคลาส:
Species_Short
Adelie       151
Gentoo       123
Chinstrap     68
Name: count, dtype: int64

สรุปผลการทดสอบ SVM แต่ละ Kernel Function
Kernel: LINEAR     | Accuracy: 100.00% (1.0000)
จำนวน Support Vectors: [11 11  3] (ต่อแต่ละคลาส)
------------------------------------------------------------
Kernel: POLY       | Accuracy: 94.19% (0.9419)
จำนวน Support Vectors: [29 28 10] (ต่อแต่ละคลาส)
------------------------------------------------------------
Kernel: RBF        | Accuracy: 100.00% (1.0000)
จำนวน Support Vectors: [21 20 10] (ต่อแต่ละคลาส)
------------------------------------------------------------

Classification Report สำหรับ RBF Kernel:
              precision    recall  f1-score   support

      Adelie       1.00      1.00      1.00        38
   Chinstrap       1.00      1.00      1.00        17
      Gentoo       1.00      1.00      1.00        31

    accuracy                           1.00        86
   macro av